In [1]:
documents = [
"""
PATIENT: DUPONT Jean
IPP: 123456
Date d'admission: 12/03/2023
Diagnostic principal: Diabète de type 2
Traitement: Metformine 850mg x2/j
Sortie prévue le: 15-03-2023
""",

"""
Patient : MARTIN   Claire
Identifiant patient: 789101
Admission le 01/11/2022
Dx principal : Insuffisance cardiaque
Traitements en cours : Lasilix 40mg; Kardegic 75 mg
Sortie: 05/11/2022
""",

"""
Nom patient: Durand Paul
IPP : 456999
Date admission : 2023-07-02
Diagnostic: HTA sévère
Traitement principal: Amlodipine
"""
]

In [13]:
import re

admission_pattern = ["admission"]
sortie_pattern = ["sortie"]
patient_pattern = ["patient", "nom"]
diagnostic_pattern = ["diagnostic", "dx"]
traitement_pattern = ["traitement"]
identity_pattern = ["ipp", "identifiant"]

DATE_PATTERN = r"\b(?:\d{2}[/-]\d{2}[/-]\d{4}|\d{4}-\d{2}-\d{2})\b"

def clean_doc(doc: str) -> str | None:
    if doc and doc.strip():
        return doc.strip()
    return None


def patient_reports(documents: list[str]) -> dict:
    clean_documents = [clean_doc(doc) for doc in documents if clean_doc(doc)]
    out = {}

    for doc in clean_documents:
        infos = doc.split("\n")

        # Initialisation par document
        name = None
        ipp = None
        date_admin = None
        date_sortie = None
        diag = None
        traitement = None

        for info in infos:
            info = info.strip()
            info_lower = info.lower()

            if any(p in info_lower for p in admission_pattern):
                match = re.search(DATE_PATTERN, info)
                if match:
                    date_admin = match.group()

            elif any(p in info_lower for p in sortie_pattern):
                match = re.search(DATE_PATTERN, info)
                if match:
                    date_sortie = match.group()

            elif any(p in info_lower for p in patient_pattern):
                if ":" in info:
                    name = info.split(":", 1)[1].strip()

            elif any(p in info_lower for p in diagnostic_pattern):
                if ":" in info:
                    diag = info.split(":", 1)[1].strip()

            elif any(p in info_lower for p in traitement_pattern):
                if ":" in info:
                    traitement = info.split(":", 1)[1].strip()

            elif any(p in info_lower for p in identity_pattern):
                if ":" in info:
                    ipp = info.split(":", 1)[1].strip()

        if name:
            out[name] = {
                "Name": name,
                "ipp": ipp,
                "admin_date": date_admin,
                "out_date": date_sortie,
                "diag": diag,
                "treatment": traitement
            }

    return out

patient_reports(documents)

{'DUPONT Jean': {'Name': 'DUPONT Jean',
  'ipp': '123456',
  'admin_date': '12/03/2023',
  'out_date': '15-03-2023',
  'diag': 'Diabète de type 2',
  'treatment': 'Metformine 850mg x2/j'},
 '789101': {'Name': '789101',
  'ipp': None,
  'admin_date': '01/11/2022',
  'out_date': '05/11/2022',
  'diag': 'Insuffisance cardiaque',
  'treatment': 'Lasilix 40mg; Kardegic 75 mg'},
 'Durand Paul': {'Name': 'Durand Paul',
  'ipp': '456999',
  'admin_date': '2023-07-02',
  'out_date': None,
  'diag': 'HTA sévère',
  'treatment': 'Amlodipine'}}